# 02 — Train PatchTST
Cell order: small-debug run, then full Weather/Electricity at T=96 and T=336.

Resumable: if Colab disconnects, re-run the cell — `resume=True` is on by default.


## Colab setup
Verify GPU, mount Drive, clone the repo, install requirements, and create the project tree on Drive.


In [ ]:
# 1. GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Mount Drive (skipped automatically when not on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/cs4782_patchtst_project'
    IN_COLAB = True
except Exception:
    PROJECT_ROOT = '.'
    IN_COLAB = False
print('PROJECT_ROOT =', PROJECT_ROOT, ' IN_COLAB =', IN_COLAB)


In [ ]:
# 3. Clone repo (Colab only). Update REPO_URL in scripts/build_notebooks.py
# and re-run that script to regenerate notebooks if the URL changes.
REPO_URL = 'https://github.com/Ash1R/ATSW64W-experiments.git'
if IN_COLAB:
    import os, subprocess
    os.chdir('/content')
    # Derive the clone directory from the URL's basename so it matches the repo name.
    REPO_DIRNAME = REPO_URL.rstrip('/').rsplit('/', 1)[-1]
    if REPO_DIRNAME.endswith('.git'):
        REPO_DIRNAME = REPO_DIRNAME[:-4]
    if not os.path.isdir(f'/content/{REPO_DIRNAME}'):
        # check=True so a bad URL fails loudly here instead of crashing the next chdir.
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIRNAME], check=True)
    os.chdir(f'/content/{REPO_DIRNAME}')
    subprocess.run(['git', 'pull'], check=False)
print('cwd =', __import__('os').getcwd())


In [ ]:
# 4. Install requirements (best-effort; resolved relative to the repo root
# regardless of where the kernel started, so headless `nbconvert` runs work).
import subprocess, sys, os
_req_dir = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_req_dir, 'requirements.txt')):
        break
    _req_dir = os.path.dirname(_req_dir)
_req = os.path.join(_req_dir, 'requirements.txt')
if os.path.isfile(_req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', _req], check=False)
else:
    print('skipping pip install — requirements.txt not found from', os.getcwd())


In [ ]:
# 5. Make the project's `code/` directory importable.
# We add `code/` itself to sys.path (not the repo root) because the
# stdlib already ships a module named `code` that the IPython kernel
# imports before this cell runs — shadowing that cleanly is messy.
# This way every import is `from utils...`, `from data...`, `from models...`.
import sys, os
# When run with `jupyter nbconvert --execute`, the kernel's cwd is
# the notebook's directory (`notebooks/`), so locate the repo root
# by walking up until we find `code/`. On Colab we already chdir'd
# into the cloned repo above.
REPO_DIR = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(REPO_DIR, 'code')):
        break
    REPO_DIR = os.path.dirname(REPO_DIR)
CODE_DIR = os.path.join(REPO_DIR, 'code')
if not os.path.isdir(CODE_DIR):
    raise RuntimeError(f'could not locate code/ from {os.getcwd()}')
os.chdir(REPO_DIR)
for p in (REPO_DIR, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
print('REPO_DIR =', REPO_DIR)
from utils.colab import ensure_dirs
subdirs = ensure_dirs(PROJECT_ROOT)
for k, v in subdirs.items():
    print(f'{k:>12}  {v}')


In [ ]:
from run_experiment import load_config
from train import train_from_config
import os
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'results')

def run(config_path, **overrides):
    cfg = load_config(config_path)
    cfg['project_root'] = PROJECT_ROOT
    cfg['output_dir'] = OUTPUT_DIR
    cfg['resume'] = True
    cfg.update(overrides)
    return train_from_config(cfg)


### Debug run (1 epoch, small model) — confirms training loop on this Colab GPU


In [ ]:
run('configs/weather_patchtst_96.yaml',
    epochs=1, d_model=64, n_heads=4, num_layers=2, d_ff=128,
    run_name='weather_patchtst_T96_DEBUG')


### Weather PatchTST T=96


In [ ]:
run('configs/weather_patchtst_96.yaml')


### Weather PatchTST T=336


In [ ]:
run('configs/weather_patchtst_336.yaml')


### Electricity PatchTST T=96


In [ ]:
run('configs/electricity_patchtst_96.yaml')


### Electricity PatchTST T=336


In [ ]:
run('configs/electricity_patchtst_336.yaml')


### (Optional) Traffic T=96 — only if memory permits


In [ ]:
import torch
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 14e9:
    run('configs/electricity_patchtst_96.yaml',
        dataset='traffic', batch_size=8,
        run_name='traffic_patchtst_T96')
else:
    print('Skipping Traffic — needs >=16GB GPU and the Traffic CSV.')
